# BioMQM — Metrics Extension

Runs two additional evaluation metrics on baseline mapped data:
1. **NLI Classifier** — `facebook/bart-large-mnli` for entailment/contradiction
2. **LLM Judge** — Qwen2.5-3B-Instruct as NLI judge for comparison
3. **Agreement Rate** — Measures label agreement between NLI Classifier and LLM Judge

## 0. Environment Setup

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
elif IN_KAGGLE:
    print('Running on Kaggle')
else:
    print('Running locally')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'sentencepiece'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

## 1. Pre-download Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
import torch

print('=== Downloading/Loading Models ===')

# NLI Classifier
NLI_MODEL = 'facebook/bart-large-mnli'
print(f'[1/2] Loading {NLI_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ NLI cached')

# Qwen (for LLM Judge)
QWEN_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
print(f'[2/2] Loading {QWEN_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ Qwen cached')

print('\n=== All models cached! ===')

## 2. Path Configuration

In [ ]:
EXTENSION_DIR = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/biomqm/metrics-extension"
EVAL_DIR = f"{EXTENSION_DIR}/evaluation"
BASELINE_DIR = f"{PROJECT_ROOT}/Qwen2.5-3B-Instruct/biomqm/baseline"

# Input: baseline mapped file
MAPPED_FILE = f"{BASELINE_DIR}/mapping/mapping.jsonl"

# Verify input exists
if os.path.exists(MAPPED_FILE):
    with open(MAPPED_FILE, 'r') as f:
        n_lines = sum(1 for _ in f)
    print(f'✓ Mapped file found ({n_lines} rows)')
else:
    print(f'✗ ERROR: Mapped file NOT found: {MAPPED_FILE}')
    print('Run the baseline pipeline first!')

## 3. NLI Classifier

Classifies answer pairs as entailment/neutral/contradiction using `facebook/bart-large-mnli`.

In [ ]:
nli_script = f'{EVAL_DIR}/nli/nli_classifier.py'

cmd = [
    sys.executable, '-u', nli_script,
    '--mapped_file_path', MAPPED_FILE,
    '--output_base_dir', EXTENSION_DIR
]

print('Running NLI Classifier...')
subprocess.run(cmd, check=True)
print('✓ NLI Classifier complete!')

## 4. LLM Judge

Uses Qwen2.5-3B-Instruct as NLI judge — compare with BART-MNLI classifier for agreement analysis.

In [ ]:
llm_script = f'{EVAL_DIR}/llm-judge/llm_judge.py'

cmd = [
    sys.executable, '-u', llm_script,
    '--mapped_file_path', MAPPED_FILE,
    '--output_base_dir', EXTENSION_DIR
]

print('Running LLM Judge...')
subprocess.run(cmd, check=True)
print('✓ LLM Judge complete!')

## 5. Agreement Rate

Computes agreement between the NLI Classifier and LLM Judge labels.
For each answer pair, both methods produce a label (entailment / neutral / contradiction).
Agreement Rate = fraction of samples where both methods agree on the label.

In [ ]:
import json
import glob

LANGUAGES = ['de', 'es', 'fr', 'ru', 'zh-CN']

# Locate NLI and LLM Judge result directories
nli_dir = os.path.join(EXTENSION_DIR, 'evaluation', 'nli')
llm_dir = os.path.join(EXTENSION_DIR, 'evaluation', 'llm-judge')

def load_labels(filepath):
    """Load labels from a JSONL results file."""
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data = json.loads(line.strip())
            label = data.get('label', data.get('nli_label', data.get('llm_label', '')))
            labels.append(label.lower().strip())
    return labels

# Look for results files and compute agreement
total_agree = 0
total_count = 0
per_lang_agreement = {}

print(f"{'='*60}")
print('Agreement Rate: NLI Classifier vs LLM Judge')
print(f"{'='*60}")

for lang in LANGUAGES:
    # Find matching result files
    nli_files = glob.glob(os.path.join(nli_dir, f'*{lang}*'))
    llm_files = glob.glob(os.path.join(llm_dir, f'*{lang}*'))

    if not nli_files or not llm_files:
        print(f'  {lang}: ⚠ Missing result files (NLI={len(nli_files)}, LLM={len(llm_files)})')
        continue

    # Use first match
    nli_labels = load_labels(nli_files[0])
    llm_labels = load_labels(llm_files[0])

    n = min(len(nli_labels), len(llm_labels))
    if n == 0:
        print(f'  {lang}: ⚠ No labels found')
        continue

    agree = sum(1 for i in range(n) if nli_labels[i] == llm_labels[i])
    rate = agree / n * 100
    per_lang_agreement[lang] = rate

    total_agree += agree
    total_count += n

    print(f'  {lang:>5}: {agree}/{n} = {rate:.1f}%')

if total_count > 0:
    overall = total_agree / total_count * 100
    print(f"\n  {'Overall':>5}: {total_agree}/{total_count} = {overall:.1f}%")
else:
    print('\n  ⚠ No results to compare — run Steps 3 and 4 first')

print(f"{'='*60}")

## Summary

Metrics Extension complete! Output locations:
- **NLI Classifier**: `evaluation/nli/`
- **LLM Judge**: `evaluation/llm-judge/`
- **Agreement Rate**: printed above (NLI vs LLM Judge label concordance per language)